In [6]:
import numpy as np
import pandas as pd
from pandas import DataFrame

# Libraries for retrieving data from the web and handling zip files
import requests, zipfile
from io import StringIO
import io

In [7]:
# Specify the url with data
url = 'https://github.com/Hernan4444/MyAnimeList-Database/archive/refs/heads/master.zip'

# Acquire data from the url
r = requests.get(url, stream=True)

# read and extract the zipfile
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()

# Load and clean the data
anime_data = pd.read_csv('MyAnimeList-Database-master/data/anime.csv')
anime_data_extracted = anime_data[anime_data['Score'] != 'Unknown'].copy()
anime_data_extracted['Score'] = pd.to_numeric(anime_data_extracted['Score'])

In [8]:


def homework(anime_data_extracted):
    # Create a working copy of the data to avoid altering the original dataframe
    df = anime_data_extracted.copy()
    
    # Step 1: Divide anime into 10 equal-frequency groups based on 'Members'
    decile_labels = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']
    df['Decile_Group'] = pd.qcut(df['Members'], q=10, labels=decile_labels)
    
    # Step 2: Group the data by our new decile labels and calculate the required stats
    # Using named aggregation makes it easy to name the new columns exactly what we want
    result = df.groupby('Decile_Group').agg(
        MeanScore=('Score', 'mean'),
        TotalMembers=('Members', 'sum')
    )
    
    # Step 3: Calculate the Cumulative Percentage
    # .cumsum() gives us the running total row-by-row
    running_total = result['TotalMembers'].cumsum()
    # .sum() gives us the grand total of all members combined
    grand_total = result['TotalMembers'].sum()
    
    # Divide the running total by the grand total, then multiply by 100 for percentage
    result['CumulativePct'] = (running_total / grand_total) * 100
    
    # Step 4: Set the index name to 'Members' so it exactly matches the example output table
    result.index.name = 'Members'
    
    return result

In [9]:
# Run this cell after completing homework().
your_answer = homework(anime_data_extracted)
display(your_answer)


,MeanScore,TotalMembers,CumulativePct
Members,,,
01,5.572864,493614,0.081881
02,5.774140,1026556,0.252165
03,5.966586,1856938,0.560194
04,6.194738,3134574,1.080156
05,6.411547,5193157,1.941596
06,6.617432,8702864,3.385224
07,6.784428,16475271,6.118138
08,6.978213,34630759,11.862679
09,7.195588,87833085,26.432408


In [10]:
case_2_data = pd.DataFrame({
    "Members": np.arange(1, 21),
    "Score": np.full(20, 5.0),
})
case_2_answer = homework(case_2_data)
display(case_2_answer)


,MeanScore,TotalMembers,CumulativePct
Members,,,
01,5.0,3,1.428571
02,5.0,7,4.761905
03,5.0,11,10.000000
04,5.0,15,17.142857
05,5.0,19,26.190476
06,5.0,23,37.142857
07,5.0,27,50.000000
08,5.0,31,64.761905
09,5.0,35,81.428571
